In [92]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [93]:
df = pd.read_csv(
    "../data/processed/cleaned_parking_data.csv"
)

df.head()

,id,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,modified_datetime,...,is_weekend,time_period,vehicle_impact,violation_count,violation_severity,obstruction_score,time_impact,congestion_impact_score,enforcement_zone,location_pressure
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00:00,2023-11-28 04:48:04.582978+00,...,False,Night,3,2,5,8,1,26.48,"18th Main Road, Block 2, Koramangala, Bengalur...",0.003236
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00:00,2023-11-24 23:00:24.115257+00,...,False,Night,3,1,3,3,1,14.12,"Sarjapura Main Road, The Grove, Janatha Colony...",0.001942
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00:00,2023-11-28 04:47:02.33776+00,...,False,Night,3,2,4,7,1,24.01,"Koramangala 2nd Block, Kormangala West, Bengal...",0.000647
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00:00,2023-11-18 04:46:57.216868+00,...,False,Morning,1,1,3,3,1,10.59,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",0.000647
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00:00,2023-11-28 02:44:50.46737+00,...,False,Night,5,1,3,3,1,27.29,BTP044 - Sagar Theatre Junction,6.828274


In [94]:
df.columns

Index(['id', 'latitude', 'longitude', 'location', 'vehicle_number',
       'vehicle_type', 'violation_type', 'offence_code', 'created_datetime',
       'modified_datetime', 'device_id', 'created_by_id', 'center_code',
       'police_station', 'data_sent_to_scita', 'junction_name',
       'data_sent_to_scita_timestamp', 'updated_vehicle_number',
       'updated_vehicle_type', 'validation_status', 'validation_timestamp',
       'hour', 'day_of_week', 'month', 'is_weekend', 'time_period',
       'vehicle_impact', 'violation_count', 'violation_severity',
       'obstruction_score', 'time_impact', 'congestion_impact_score',
       'enforcement_zone', 'location_pressure'],
      dtype='str')

In [95]:
df.shape

(298450, 34)

In [96]:
#current congestion distribution
df["congestion_impact_score"].describe()

count    298450.00000
mean         16.24850
std           6.73695
min           8.12000
25%          11.65000
50%          14.24000
75%          20.23000
max         100.00000
Name: congestion_impact_score, dtype: float64

In [97]:
#create hotspot aggregation table
hotspots = (
    df.groupby("enforcement_zone")
    .agg(
        violation_count=("id", "count"),
        avg_congestion_score=(
            "congestion_impact_score",
            "mean"
        ),
        max_congestion_score=(
            "congestion_impact_score",
            "max"
        ),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .reset_index()
)

hotspots.head()

,enforcement_zone,violation_count,avg_congestion_score,max_congestion_score,latitude,longitude
0,"1, 12th Main Road, Sector 6, HSR Layout, Benga...",1,10.59,10.59,12.915555,77.636899
1,"1, 14th Main Road, Sector 7, HSR Layout, Benga...",6,8.12,8.12,12.912080,77.638002
2,"1, 7th C Cross Road, Maistripalaya, Kormangala...",1,11.65,11.65,12.932624,77.623365
3,"1/1, East Park Road, Sadashivanagar, Sadashiva...",2,13.77,14.12,13.008411,77.570690
4,"100 Feet Road, 3rd Block, Sir MV Layout, Benga...",1,31.42,31.42,12.951517,77.477497


In [98]:
#top violating hotspots
hotspots.sort_values(
    "violation_count",
    ascending=False
).head(10)

,enforcement_zone,violation_count,avg_congestion_score,max_congestion_score,latitude,longitude
2832,BTP051 - Safina Plaza Junction,15449,26.205244,55.42,12.981212,77.608695
2854,BTP082 - KR Market Junction,11538,23.064573,64.20,12.964424,77.577241
2824,BTP040 - Elite Junction,10718,21.418024,67.34,12.976597,77.576602
2828,BTP044 - Sagar Theatre Junction,10549,21.741096,40.35,12.974939,77.578896
2952,BTP211 - Central Street Junction,5388,18.315767,46.23,12.983371,77.603454
2837,BTP058 - Subbanna Junction,5189,19.975598,46.04,12.979022,77.578661
2814,BTP027 - Modi Bridge Junction,4584,14.617805,45.49,12.998897,77.549416
2808,BTP020 - Hosahalli Metro Station,4101,13.582958,39.40,12.974144,77.545294
7225,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",4090,21.624005,67.28,13.185700,77.680037
2836,BTP057 - Anand Rao Junction,3935,17.021393,46.66,12.979395,77.574388


In [99]:
#creating frequency normalization
hotspots["frequency_score"] = (
    np.log1p(hotspots["violation_count"])/np.log1p(hotspots["violation_count"].max())*100
)

In [100]:
#AI priority score
hotspots["priority_score"] = (

    hotspots["frequency_score"] * 0.5 + hotspots["avg_congestion_score"] * 0.3 + hotspots["max_congestion_score"] * 0.2

)

In [101]:
#Assigning risk levels
def assign_risk(score):

    if score >= 50:
        return "Critical"

    elif score >= 40:
        return "High"

    elif score >= 25:
        return "Medium"

    else:
        return "Low"


hotspots["risk_level"] = (
    hotspots["priority_score"]
    .apply(assign_risk)
)

In [102]:
hotspots["risk_level"].value_counts()

risk_level
Low         6251
Medium      1152
High         154
Critical      45
Name: count, dtype: int64

In [103]:
hotspots.sort_values(
    "priority_score",
    ascending=False
).head(15)

,enforcement_zone,violation_count,avg_congestion_score,max_congestion_score,latitude,longitude,frequency_score,priority_score,risk_level
2832,BTP051 - Safina Plaza Junction,15449,26.205244,55.42,12.981212,77.608695,100.000000,68.945573,Critical
2854,BTP082 - KR Market Junction,11538,23.064573,64.20,12.964424,77.577241,96.973920,68.246332,Critical
2824,BTP040 - Elite Junction,10718,21.418024,67.34,12.976597,77.576602,96.209670,67.998242,Critical
5775,"New Horizon College Road, New Horizon College ...",3785,19.144642,91.71,12.933819,77.690900,85.419950,66.795368,Critical
2798,"BTP001 - 10th Cross, Dr. Rajkumar Road",2812,14.938802,100.00,13.007811,77.554653,82.340144,65.651712,Critical
7225,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",4090,21.624005,67.28,13.185700,77.680037,86.223231,63.054817,Critical
2828,BTP044 - Sagar Theatre Junction,10549,21.741096,40.35,12.974939,77.578896,96.044907,62.614782,Critical
2837,BTP058 - Subbanna Junction,5189,19.975598,46.04,12.979022,77.578661,88.690160,59.545759,Critical
7221,Unknown,1709,16.865407,79.22,12.971093,77.625801,77.179549,59.493396,Critical
2952,BTP211 - Central Street Junction,5388,18.315767,46.23,12.983371,77.603454,89.080255,59.280858,Critical


In [104]:
hotspots = hotspots[
    [
        "enforcement_zone",
        "latitude",
        "longitude",

        "violation_count",
        "violation_pressure",
        "frequency_score",

        "avg_congestion_score",
        "max_congestion_score",

        "priority_score",
        "risk_level"
    ]
]

KeyError: "['violation_pressure'] not in index"

In [ ]:
hotspots.to_csv(
    "../outputs/hotspot_rankings.csv",
    index=False
)

In [ ]:
import os

os.path.exists(
    "../outputs/hotspot_rankings.csv"
)

True

# Enforcement Officer Allocation Recommendation

In [ ]:
#create enforcement demand score
hotspots["enforcement_demand_score"] = (

    hotspots["priority_score"] * 0.5
    +
    hotspots["violation_pressure"] * 0.3
    +
    hotspots["avg_congestion_score"] * 0.2

)

In [ ]:
available_officers = 1000 # User input from dashboard

deploy_zones = (
    hotspots
    .sort_values(
        "enforcement_demand_score",
        ascending=False
    )
    .copy()
)

In [ ]:
deploy_zones["demand_percentage"] = (
    deploy_zones["enforcement_demand_score"]
    /
    deploy_zones["enforcement_demand_score"].sum()
)

In [ ]:
deploy_zones["cumulative_demand"] = (
    deploy_zones["demand_percentage"]
    .cumsum()
)

In [ ]:
deploy_zones = deploy_zones[
    deploy_zones["cumulative_demand"] <= 0.30
].copy()

In [ ]:
deploy_zones.shape

(1312, 13)

In [ ]:
#allocating officers
deploy_zones["allocation_weight"] = (
    deploy_zones["enforcement_demand_score"]
    /
    deploy_zones["enforcement_demand_score"].sum()
)

In [ ]:
deploy_zones["recommended_officers"] = (
    deploy_zones["allocation_weight"]
    *
    available_officers
).round().astype(int)

In [ ]:
deploy_zones.loc[
    deploy_zones["recommended_officers"] == 1,
    "recommended_officers"
] = 1

In [ ]:
deploy_zones.shape

(1312, 15)

In [ ]:
deploy_zones["recommended_officers"].sum()

np.int64(1322)

In [ ]:
deploy_zones[
[
"enforcement_zone",
"enforcement_demand_score",
"risk_level",
"recommended_officers"
]
].sort_values(
    "recommended_officers",
    ascending=False
).head(20)

,enforcement_zone,enforcement_demand_score,risk_level,recommended_officers
2832,BTP051 - Safina Plaza Junction,69.713836,Critical,3
2854,BTP082 - KR Market Junction,61.141414,Critical,2
2824,BTP040 - Elite Junction,59.095723,Critical,2
2828,BTP044 - Sagar Theatre Junction,56.140431,Critical,2
5775,"New Horizon College Road, New Horizon College ...",44.576602,Critical,2
2837,BTP058 - Subbanna Junction,43.844380,Critical,2
7225,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",43.794471,Critical,2
2952,BTP211 - Central Street Junction,43.766395,Critical,2
2798,"BTP001 - 10th Cross, Dr. Rajkumar Road",41.274164,Critical,2
2814,BTP027 - Modi Bridge Junction,40.418067,Critical,1


In [ ]:
deploy_zones.to_csv(
    "../outputs/enforcement_plan.csv",
    index=False
)

# Peak Hour Forecasting Engine

In [105]:
df["hour"].head()

0     0
1    22
2     0
3     6
4     4
Name: hour, dtype: int64

In [106]:
df["hour"].value_counts().sort_index()

hour
0     21762
1     17155
2     24770
3     25709
4     29102
5     34085
6     26890
7     14608
8      8556
9      3145
10      518
11      577
12      219
13       56
14       16
15       66
16      416
17      818
18     1971
19    10713
20    11834
21    19763
22    22840
23    22861
Name: count, dtype: int64

In [107]:
hourly_risk = (
    df.groupby(
        [
            "enforcement_zone",
            "hour"
        ]
    )
    .agg(
        violation_count=(
            "id",
            "count"
        ),
        
        avg_congestion_score=(
            "congestion_impact_score",
            "mean"
        )
    )
    .reset_index()
)

hourly_risk.head()

,enforcement_zone,hour,violation_count,avg_congestion_score
0,"1, 12th Main Road, Sector 6, HSR Layout, Benga...",4,1,10.59
1,"1, 14th Main Road, Sector 7, HSR Layout, Benga...",4,6,8.12
2,"1, 7th C Cross Road, Maistripalaya, Kormangala...",22,1,11.65
3,"1/1, East Park Road, Sadashivanagar, Sadashiva...",1,1,13.42
4,"1/1, East Park Road, Sadashivanagar, Sadashiva...",5,1,14.12


In [113]:
hourly_risk["hourly_frequency_score"] = (
    np.log1p(hourly_risk["violation_count"])
    /
    np.log1p(hourly_risk["violation_count"].max())
    *
    100
)

In [116]:
hourly_risk["hourly_risk_score"] = (

    hourly_risk["hourly_frequency_score"] * 0.7
    +
    hourly_risk["avg_congestion_score"] * 0.3
)

In [117]:
peak_hours = (
    hourly_risk
    .sort_values(
        "hourly_risk_score",
        ascending=False
    )
    .groupby(
        "enforcement_zone"
    )
    .head(1)
)

In [120]:
peak_hours["recommended_time_window"] = (
    peak_hours["hour"]
    .astype(str)
    .str.zfill(2)
    +
    ":00 - "
    +
    (
        (peak_hours["hour"] + 1) % 24
    )
    .astype(str)
    .str.zfill(2)
    +
    ":00"
)

In [121]:
peak_hours[
    [
        "enforcement_zone",
        "recommended_time_window",
        "violation_count",
        "hourly_risk_score"
    ]
].sort_values(
    "hourly_risk_score",
    ascending=False
).head(20)

,enforcement_zone,recommended_time_window,violation_count,hourly_risk_score
7578,BTP051 - Safina Plaza Junction,05:00 - 06:00,2972,77.811940
7912,BTP082 - KR Market Junction,19:00 - 20:00,1788,73.590413
7510,BTP044 - Sagar Theatre Junction,03:00 - 04:00,1677,71.387145
7443,BTP040 - Elite Junction,03:00 - 04:00,1717,71.330642
7662,BTP058 - Subbanna Junction,19:00 - 20:00,1390,70.406455
16530,"MBT Road, Devasandra Junction, KR Puram, Benga...",19:00 - 20:00,679,63.794636
9167,BTP211 - Central Street Junction,05:00 - 06:00,712,62.879731
18826,"New Horizon College Road, New Horizon College ...",23:00 - 00:00,654,62.410079
24273,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",19:00 - 20:00,459,62.319393
7880,"BTP080 - NR Road, SP Road Junction",04:00 - 05:00,635,60.963056


In [122]:
peak_hours.to_csv(
    "../outputs/peak_hour_forecast.csv",
    index=False
)

In [123]:
import os

os.path.exists(
    "../outputs/peak_hour_forecast.csv"
)

True